# Vedic Parallel - Difficulty Assignment (Hindi to Sanskrit / Kannada)

Assigns `difficulty` (Easy / Medium / Hard) to 300 sampled rows from **either**
vedic pair - set `DATASET` in Cell 3:

| DATASET | File | Rows | Pair |
|---|---|---|---|
| `"hi_sa"` | `vedic_hi_sa.jsonl` | 1983 | Hindi to Sanskrit |
| `"hi_kn"` | `vedic_hi_kn.jsonl` | 1935 | Hindi to Kannada |

Output paths and progress directories derive from that name, so the two runs
never overwrite each other.

**Task.** `question` is Hindi in Devanagari - commentary on Vedic texts and
Paninian grammar - and `answer` is its Sanskrit or Kannada translation. Scored
with **chrF++**, which is what the files declare.

**Two-pass design**, unchanged from the santham notebook:

1. Score every row with every model, storing the raw chrF++ value.
2. Derive a threshold (Cell 9); a model passes a row when it clears it, and the
   three votes sum as usual:

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**The two pairs behave very differently, and the notebook does not hide it.**
Hindi and Kannada use different scripts, so copying the Hindi input scores ~0
chrF++ and the binding reference is the unrelated-translation floor - exactly
like santham. But **Hindi and Sanskrit share the Devanagari script and a large
amount of vocabulary**, so for `hi_sa` copying the input scores *substantially
above zero*. Cell 4 measures the floor for whichever pair you selected, and
Cell 9 anchors to the higher of the two floors, so `hi_sa` is protected from a
threshold that would pass a model which merely echoed the Hindi.

**Output.** `vedic_<pair>_difficulty.jsonl` - all 14 schema fields with
`difficulty` filled in, plus an audit file with every model's translation.

### Cell 1 - Install dependencies and authenticate

Adds `sacrebleu` to the usual stack - it provides the reference implementation
of chrF++ with a reproducible signature, which is safer than hand-rolling
character n-gram scoring.

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; only Mistral is
open. Accept each licence on huggingface.co, create a **read** token, then add
it in Colab via the **key icon** as a secret named `HF_TOKEN`. Use the secret
rather than pasting the token into a cell.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sacrebleu

import sacrebleu
print("sacrebleu", sacrebleu.__version__)

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.9 MB/s eta 0:00:00
sacrebleu 2.6.0
HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: models are downloaded once, quantised to 4-bit and
saved here, so later runs skip the download entirely.

`DRIVE_OK` records whether the mount actually succeeded - later cells check it
rather than assuming. If you see **"credential propagation was unsuccessful"**,
the auth popup did not complete: re-run and finish it, allow pop-ups and
third-party cookies for `colab.research.google.com`, or mount from the Files
sidebar. Without Drive everything still runs, just uncached.

In [2]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models


### Cell 3 - Configuration

- `DATASET` - `"hi_sa"` or `"hi_kn"`. Everything else derives from it, so
  switching pairs is a one-word change and the runs keep separate progress
  files.
- `THRESHOLD_MODE` - `"auto_median"` by default. `"floor_margin"` anchors to
  the higher of the two measured floors plus `FLOOR_MARGIN`, which is the safer
  choice for `hi_sa` where Hindi and Sanskrit overlap heavily.
- `SET_EVAL_METRIC` - `None` keeps the file's own `chrF++`.
- `MODELS` - the same three judges as every other split. None is strong at
  Sanskrit; Kannada is moderately resourced for them. Read Cell 11 before
  drawing conclusions from a Hard-heavy result.

In [3]:
import gc
import re
import json
import random
import shutil
import statistics
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- which vedic pair ----
DATASET = "hi_sa"                  # "hi_sa" or "hi_kn"

_FILES  = {"hi_sa": "vedic_hi_sa.jsonl", "hi_kn": "vedic_hi_kn.jsonl"}
_TARGET = {"hi_sa": "Sanskrit",         "hi_kn": "Kannada"}
assert DATASET in _FILES, "DATASET must be one of {}".format(list(_FILES))
TARGET_LANG = _TARGET[DATASET]

# ---- paths (derived, so the two runs never collide) ----
INPUT_FILE  = _FILES[DATASET]
OUTPUT_FILE = "vedic_{}_difficulty.jsonl".format(DATASET)
AUDIT_FILE  = "vedic_{}_audit.jsonl".format(DATASET)
PROG_DIR    = "judge_progress_{}".format(DATASET)

# ---- sampling ----
N_ROWS = 300
SEED   = 42
MIN_REF_WORDS = 3        # drop rows whose reference translation is junk

# ---- thresholding ----
THRESHOLD_MODE  = "auto_median"     # auto_median | floor_margin | fixed
FLOOR_MARGIN    = 0.15              # used by floor_margin, above RAND_FLOOR
FIXED_THRESHOLD = 0.50

# ---- generation ----
MAX_NEW_TOKENS = 256
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = None              # None keeps the file's value; e.g. "chrf++"

# ---- the three judges ----
USE_INSTRUCT = True

REPOS = {
    True: {
        "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # ungated
        "llama":   "meta-llama/Llama-3.1-8B-Instruct",     # GATED
        "gemma":   "google/gemma-2-9b-it",                 # GATED
    },
    False: {
        "mistral": "mistralai/Mistral-7B-v0.3",
        "llama":   "meta-llama/Llama-3.1-8B",
        "gemma":   "google/gemma-2-9b",
    },
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("dataset: {} -> {}  (Hindi -> {})".format(DATASET, INPUT_FILE, TARGET_LANG))
print("Judges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached in Drive" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

dataset: hi_sa -> vedic_hi_sa.jsonl  (Hindi -> Sanskrit)
Judges (instruct):
  mistral  mistralai/Mistral-7B-Instruct-v0.3         cached in Drive
  llama    meta-llama/Llama-3.1-8B-Instruct           cached in Drive
  gemma    google/gemma-2-9b-it                       cached in Drive

device: cuda


### Cell 4 - Load, sample, and measure the reference floors

Samples 200 rows under a fixed seed, so the same rows return on every run -
which is what makes the resume logic safe across sessions.

Then two model-free reference points:

- **Copy the Hindi input.** For `hi_kn` this is near zero: Devanagari and
  Kannada share no characters. For `hi_sa` it is **not** near zero - Hindi and
  Sanskrit share the Devanagari script and a great deal of vocabulary, so
  echoing the input already earns real credit. This is the number that
  constrains the threshold for that pair.
- **An unrelated row's translation** - the true wrong-answer level, which still
  shares script and common morphology with the reference.

Cell 9 anchors to whichever of the two is higher, so neither pair can end up
with a threshold that passes a model which translated nothing.

`usable()` checks the target script for the selected pair - Kannada for
`hi_kn`, Devanagari for `hi_sa`. Getting this wrong silently empties the sample.

In [4]:
import sacrebleu
_CHRF = sacrebleu.CHRF(word_order=2)          # word_order=2 makes this chrF++


def chrf_pp(hypothesis, reference):
    if not hypothesis or not hypothesis.strip():
        return 0.0
    return _CHRF.sentence_score(hypothesis, [reference]).score / 100.0


with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))

_DEVA_RE   = re.compile(r"[ऀ-ॿ]")
_KANNADA   = re.compile(r"[ಀ-೿]")
_TARGET_RE = {"hi_sa": _DEVA_RE, "hi_kn": _KANNADA}[DATASET]


def usable(row):
    # the reference must be in the target script of the selected pair, and must
    # not be a serialised missing value - the source corpus contained "nan"
    ref = str(row.get("answer") or "").strip()
    src = str(row.get("question") or "").strip()
    if not src or ref.lower() in ("", "nan", "none"):
        return False
    return len(ref.split()) >= MIN_REF_WORDS and bool(_TARGET_RE.search(ref))


usable_rows = [r for r in all_rows if usable(r)]
print("usable rows: {}/{}  ({} dropped: reference not usable Tamil)".format(
    len(usable_rows), len(all_rows), len(all_rows) - len(usable_rows)))
assert len(usable_rows) >= N_ROWS, "not enough usable rows to sample from"

random.seed(SEED)
sample = random.sample(usable_rows, N_ROWS)

lens = sorted(len(r["question"].split()) for r in sample)
print("Sampled {} rows | source words: min {}, median {}, max {}".format(
    len(sample), lens[0], lens[len(lens) // 2], lens[-1]))

# ---- model-free floors ----
copy_scores = [chrf_pp(r["question"], r["answer"]) for r in sample]
rand_scores = [chrf_pp(sample[(i + 7) % len(sample)]["answer"], r["answer"])
               for i, r in enumerate(sample)]

COPY_FLOOR = statistics.mean(copy_scores)
RAND_FLOOR = statistics.mean(rand_scores)

print("\nReference floors (chrF++, no model involved):")
print("  copy the Hindi input verbatim    : {:.1%}".format(COPY_FLOOR))
print("  an unrelated {:<8} translation : {:.1%}".format(TARGET_LANG, RAND_FLOOR))
print("  a perfect translation            : 100.0%")
BINDING_FLOOR = max(COPY_FLOOR, RAND_FLOOR)
print("\n  binding floor for this pair: {:.1%}".format(BINDING_FLOOR))
print("  usable range for a threshold: roughly {:.0%} to 100%".format(BINDING_FLOOR))
if DATASET == "hi_sa" and COPY_FLOOR > RAND_FLOOR:
    print("\n  NOTE: copying the Hindi input outscores an unrelated Sanskrit")
    print("  sentence. Hindi and Sanskrit share Devanagari and much vocabulary,")
    print("  so a model can score well by barely changing the input. Cell 9")
    print("  anchors to this floor - do not lower the threshold beneath it.")
if DATASET == "hi_kn" and COPY_FLOOR > 0.10:
    print("\n  WARNING: copying the Hindi input scores {:.1%}. Devanagari and".format(COPY_FLOOR))
    print("  Kannada share no characters, so this should be near zero - check")
    print("  that Kannada text has not leaked into the question field.")

print("\n--- example row ---")
print("  hindi    :", " ".join(sample[0]["question"].split())[:92])
print("  {:<9}:".format(TARGET_LANG.lower()), " ".join(str(sample[0]["answer"]).split())[:92])
print("  subcategory:", sample[0]["subcategory"])

Loaded 1983 rows from vedic_hi_sa.jsonl
usable rows: 1905/1983  (78 dropped: reference not usable Tamil)
Sampled 300 rows | source words: min 1, median 11, max 56

Reference floors (chrF++, no model involved):
  copy the Hindi input verbatim    : 36.4%
  an unrelated Sanskrit translation : 9.6%
  a perfect translation            : 100.0%

  binding floor for this pair: 36.4%
  usable range for a threshold: roughly 36% to 100%

  NOTE: copying the Hindi input outscores an unrelated Sanskrit
  sentence. Hindi and Sanskrit share Devanagari and much vocabulary,
  so a model can score well by barely changing the input. Cell 9
  anchors to this floor - do not lower the threshold beneath it.

--- example row ---
  hindi    : संहितायां स्वरिताद्‌ अनुदात्तानाम्‌ एकश्रुतिः स्याद ये सूत्र में आये पदों का अन्वय है।
  sanskrit : संहितायां स्वरिताद्‌ अनुदात्तानाम्‌ एकश्रुतिः स्यादिति सूत्रगतपदानाम्‌ अन्वयः।
  subcategory: vedic_literature


### Cell 5 - Build the translation prompt

Few-shot examples are drawn from **outside** the 200-row sample, so no scored
row ever has its reference shown to the model. They are picked deterministically
from `SEED`.

They carry more weight for `hi_sa` than anywhere else in this corpus. Hindi and
Sanskrit are close enough that a model can "translate" by lightly editing the
input and still score well on chrF++; the examples are what show it the degree
of transformation the references actually apply - full Sanskrit morphology,
sandhi and case endings rather than Hindi with Sanskrit vocabulary.

This corpus has a single subcategory, so all rows share one few-shot set.

In [5]:
_INSTR = {
    "hi_sa": (
        "Translate Hindi into classical Sanskrit.\n\n"
        "The Hindi comes from commentary on Vedic texts and Paninian grammar. "
        "Produce proper Sanskrit - correct case endings, verb forms and sandhi "
        "- not Hindi with Sanskrit vocabulary. Both languages use Devanagari, "
        "so the output must differ from the input in morphology, not just in "
        "word choice.\n\n"
        "Output only the Sanskrit - no explanation, no repetition of the Hindi."
    ),
    "hi_kn": (
        "Translate Hindi into Kannada.\n\n"
        "The Hindi comes from commentary on Vedic texts and Paninian grammar. "
        "Write formal literary Kannada in the Kannada script. Keep technical "
        "Sanskrit terms and proper names in their usual Kannada forms.\n\n"
        "Output only the Kannada translation - no transliteration into Roman "
        "or Devanagari, no explanation, no repetition of the Hindi."
    ),
}
INSTRUCTIONS = _INSTR[DATASET]


def flat(text):
    return " ".join(str(text).split())


def pick_fewshot(k=3, subcat=None):
    used = {r["id"] for r in sample}
    pool = [r for r in all_rows
            if r["id"] not in used
            and 6 <= len(r["question"].split()) <= 22
            and usable(r)
            and (subcat is None or r["subcategory"] == subcat)]
    random.Random(SEED + 1).shuffle(pool)
    return pool[:k]


# parallel mixes prose and poetry; match the register where we can
_SUBCATS = sorted({r["subcategory"] for r in sample})
FEWSHOT_BY_SUBCAT = {s: (pick_fewshot(subcat=s) or pick_fewshot()) for s in _SUBCATS}
FEWSHOT = FEWSHOT_BY_SUBCAT[_SUBCATS[0]]


def shots_for(row):
    return FEWSHOT_BY_SUBCAT.get(row["subcategory"], FEWSHOT)


def build_completion(source, shots=None):
    shots = FEWSHOT if shots is None else shots
    text = INSTRUCTIONS + "\n"
    for r in shots:
        text += "\nHindi: {}\n{}: {}\n".format(
            flat(r["question"]), TARGET_LANG, flat(r["answer"]))
    text += "\nHindi: {}\n{}:".format(flat(source), TARGET_LANG)
    return text


def build_chat_messages(source, shots=None):
    shots = FEWSHOT if shots is None else shots
    msgs = [{"role": "system", "content": INSTRUCTIONS}]
    for r in shots:
        msgs.append({"role": "user", "content": flat(r["question"])})
        msgs.append({"role": "assistant", "content": flat(r["answer"])})
    msgs.append({"role": "user", "content": flat(source)})
    return msgs


for s in _SUBCATS:
    print("Few-shot for {} ({}), all from OUTSIDE the sample:".format(
        s, len(FEWSHOT_BY_SUBCAT[s])))
    for r in FEWSHOT_BY_SUBCAT[s]:
        print("   {} | {}".format(r["id"], flat(r["question"])[:58]))

print("\n" + "=" * 66)
print(build_completion(sample[0]["question"], shots_for(sample[0])))
print("=" * 66)
print("[reference: {}]".format(flat(sample[0]["answer"])))

Few-shot for vedic_literature (3), all from OUTSIDE the sample:
   vedic_003913 | उदाहरण -यहाँ उदाहरण है तावत्‌ कृष्ण चतुर्दशी।
   vedic_003331 | कपर्दः अस्यास्तीति विग्रह करने पर अत्‌ इनिठनौ इससे इनिप्रत
   vedic_000817 | इसको कल्पतरु में इस प्रकार से प्रतिपादित किया है- निर्विशे

Translate Hindi into classical Sanskrit.

The Hindi comes from commentary on Vedic texts and Paninian grammar. Produce proper Sanskrit - correct case endings, verb forms and sandhi - not Hindi with Sanskrit vocabulary. Both languages use Devanagari, so the output must differ from the input in morphology, not just in word choice.

Output only the Sanskrit - no explanation, no repetition of the Hindi.

Hindi: उदाहरण -यहाँ उदाहरण है तावत्‌ कृष्ण चतुर्दशी।
Sanskrit: उदाहरणम्‌ - अत्रोदाहरणं तावत्‌ कृष्णचतुर्दशी इति।

Hindi: कपर्दः अस्यास्तीति विग्रह करने पर अत्‌ इनिठनौ इससे इनिप्रत्यय करने पर निष्पन्न करपर्दिन्‌-शब्द का षष्ठी एकवचन में रूप है।
Sanskrit: कपर्दः अस्यास्तीति विग्रहे अत इनिठनौ इत्यनेन इनिप्रत्यये निष

### Cell 6 - Generate translations and score them

Greedy decoding (`do_sample=False`) so results are reproducible, with
`max_new_tokens` scaled to the source length rather than fixed.

`clean_translation` strips the labels models prepend (`Tamil:`, `Translation:`)
and cuts anything from a following `Sanskrit:` marker, which base models emit as
they continue the few-shot pattern. Left in, that boilerplate would drag chrF++
down for a *formatting* reason and be mistaken for a bad translation.

It does **not** strip Devanagari. A model that answers in Sanskrit, or
transliterates rather than translating, has failed the task and should score
near zero - Cell 11 reports how often that happens, since a high rate is a
prompting problem rather than a difficulty signal.

In [6]:
_LABEL = re.compile(
    r"^\s*(here(?:\s+is|'s)?\s+the\s+)?(sanskrit\s+|kannada\s+)?translation\s*[:\-]\s*",
    re.I)
_LABEL2 = re.compile(r"^\s*(sanskrit|kannada|hindi)\s*[:\-]\s*", re.I)


def wrong_script(text):
    # hi_kn: answering in Devanagari means it did not translate at all.
    # hi_sa: both sides are Devanagari, so this check does not apply.
    if DATASET != "hi_kn":
        return False
    return bool(_DEVA_RE.search(text or "")) and not _KANNADA.search(text or "")


def clean_translation(text):
    t = (text or "").strip()
    # a base model keeps going with the next few-shot block - cut it there
    t = re.split(r"\n\s*Hindi\s*:", t)[0]
    for line in t.split("\n"):
        line = line.strip()
        if not line:
            continue
        line = _LABEL.sub("", line)
        line = _LABEL2.sub("", line)
        line = line.strip().strip('"')
        if line:
            return line
    return ""


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


@torch.no_grad()
def translate(model, tokenizer, source, shots=None):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(source, shots)
    else:
        msgs = build_chat_messages(source, shots)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    budget = min(4 * len(str(source).split()) + 48, MAX_NEW_TOKENS)

    out = model.generate(**inputs,
                         max_new_tokens=budget,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_translation(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


# quick check that the cleaner behaves
for raw, want in [
    ("Translation: atha", "atha"),
    ("Here is the Sanskrit translation: hi", "hi"),
    ("Kannada: good\nHindi: aur", "good"),
    ('"quoted output"', "quoted output"),
    ("", ""),
]:
    got = clean_translation(raw)
    print("  clean {!r:<44} -> {!r}".format(raw, got))
    assert got == want, (raw, got, want)
print("\nGeneration and scoring functions defined")

  clean 'Translation: atha'                          -> 'atha'
  clean 'Here is the Sanskrit translation: hi'       -> 'hi'
  clean 'Kannada: good\nHindi: aur'                  -> 'good'
  clean '"quoted output"'                            -> 'quoted output'
  clean ''                                           -> ''

Generation and scoring functions defined


### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements download-once: if `models/<name>_4bit` exists in Drive
it is loaded directly (already 4-bit, so passing a fresh `BitsAndBytesConfig`
would conflict and is omitted); otherwise the repo is downloaded, quantised,
and saved to Drive for next time. `trust_remote_code` stays off - repo-shipped
modelling code is often written against an older transformers API.

`run_model` stores the **raw chrF++ score and the translation itself**, never a
pass/fail. The threshold is applied later in Cell 9, which is what lets you
re-derive difficulty for free. Each finished batch is appended to
`judge_progress/<model>.jsonl` before the next begins, so a disconnect costs at
most `BATCH_SIZE` rows.

In [7]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog_file = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    # ---- 1. resume from whatever is already on disk ----
    done = {}
    if os.path.exists(prog_file):
        with open(prog_file, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already scored".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    # ---- 2. load (from Drive if cached, else download and cache) ----
    model, tokenizer = load_model(spec)

    # ---- 3. translate in batches, saving after each one ----
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_start in range(0, len(remaining), BATCH_SIZE):
        batch     = remaining[batch_start : batch_start + BATCH_SIZE]
        batch_num = batch_start // BATCH_SIZE + 1

        batch_results = []
        for row in batch:
            hyp = translate(model, tokenizer, row["question"], shots_for(row))
            batch_results.append({
                "id":      row["id"],
                "chrf":    chrf_pp(hyp, str(row["answer"])),
                "n_words": len(hyp.split()),
                "wrong_script": int(wrong_script(hyp)),
                "output":  hyp,
            })

        with open(prog_file, "a", encoding="utf-8") as f:
            for item in batch_results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        done.update({item["id"]: item for item in batch_results})
        mean_chrf = sum(v["chrf"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | mean chrF++ {:.1%}".format(
            batch_num, total_batches, len(done), len(rows), mean_chrf))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

Runner defined


### Cell 8 - Run all three models

One model at a time - loaded, scored, unloaded - so peak VRAM stays near 6 GB
rather than the ~17 GB all three would need together.

This is the long cell. Every row generates a full sentence, so budget roughly
**8-12 min per model**, plus downloads on the first run. Safe to re-run:
anything already scored is skipped.

In [8]:
preds = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    preds[spec["name"]] = run_model(spec, sample)

print("\nAll models done")


=== mistral ===
  loading /drive/MyDrive/models/mistral_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 4.14GB | prompt style: chat
  batch 1/12 saved - 25/300 rows | mean chrF++ 36.8%
  batch 2/12 saved - 50/300 rows | mean chrF++ 38.9%
  batch 3/12 saved - 75/300 rows | mean chrF++ 37.1%
  batch 4/12 saved - 100/300 rows | mean chrF++ 35.9%
  batch 5/12 saved - 125/300 rows | mean chrF++ 36.0%
  batch 6/12 saved - 150/300 rows | mean chrF++ 36.3%
  batch 7/12 saved - 175/300 rows | mean chrF++ 36.2%
  batch 8/12 saved - 200/300 rows | mean chrF++ 35.5%
  batch 9/12 saved - 225/300 rows | mean chrF++ 36.3%
  batch 10/12 saved - 250/300 rows | mean chrF++ 36.3%
  batch 11/12 saved - 275/300 rows | mean chrF++ 36.2%
  batch 12/12 saved - 300/300 rows | mean chrF++ 36.6%
  mistral complete

=== llama ===
  loading /drive/MyDrive/models/llama_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 5.71GB | prompt style: chat


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  batch 1/12 saved - 25/300 rows | mean chrF++ 44.4%
  batch 2/12 saved - 50/300 rows | mean chrF++ 47.7%
  batch 3/12 saved - 75/300 rows | mean chrF++ 45.3%
  batch 4/12 saved - 100/300 rows | mean chrF++ 44.6%
  batch 5/12 saved - 125/300 rows | mean chrF++ 45.0%
  batch 6/12 saved - 150/300 rows | mean chrF++ 44.4%
  batch 7/12 saved - 175/300 rows | mean chrF++ 43.8%
  batch 8/12 saved - 200/300 rows | mean chrF++ 43.0%
  batch 9/12 saved - 225/300 rows | mean chrF++ 43.5%
  batch 10/12 saved - 250/300 rows | mean chrF++ 43.3%
  batch 11/12 saved - 275/300 rows | mean chrF++ 43.0%
  batch 12/12 saved - 300/300 rows | mean chrF++ 43.2%
  llama complete

=== gemma ===
  loading /drive/MyDrive/models/gemma_4bit [Drive cache (already 4-bit), attn=eager]


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

  ready | VRAM: 6.14GB | prompt style: chat
  batch 1/12 saved - 25/300 rows | mean chrF++ 40.7%
  batch 2/12 saved - 50/300 rows | mean chrF++ 44.0%
  batch 3/12 saved - 75/300 rows | mean chrF++ 42.9%
  batch 4/12 saved - 100/300 rows | mean chrF++ 41.9%
  batch 5/12 saved - 125/300 rows | mean chrF++ 43.3%
  batch 6/12 saved - 150/300 rows | mean chrF++ 42.9%
  batch 7/12 saved - 175/300 rows | mean chrF++ 42.6%
  batch 8/12 saved - 200/300 rows | mean chrF++ 42.1%
  batch 9/12 saved - 225/300 rows | mean chrF++ 42.3%
  batch 10/12 saved - 250/300 rows | mean chrF++ 42.2%
  batch 11/12 saved - 275/300 rows | mean chrF++ 41.8%
  batch 12/12 saved - 300/300 rows | mean chrF++ 42.1%
  gemma complete

All models done


### Cell 9 - Find the threshold

The pass mark is **derived from the data**, not guessed. The cell prints the
score distribution per model and pooled, places the model-free floors from Cell
4 alongside, and shows what every candidate threshold does to the split.

`"auto_median"` (the default) takes the median of all pooled model scores and
gives the most balanced Easy/Medium/Hard spread. It is safe here because the
do-nothing floor is ~1% - a median cannot land below it. That was a real risk
in the Hinglish notebooks, where copying the input already scored 38-51%.

`"floor_margin"` anchors to the **unrelated-translation floor** plus
`FLOOR_MARGIN`, asking "did this model beat an unrelated Tamil sentence by a
real margin?" That is the meaningful floor here; the copy-source figure is near
zero and only serves as a wiring check.

Warnings fire if the chosen threshold falls at or below the unrelated floor,
sits so high that only near-exact matches pass, or empties a difficulty band.

Watch for one model whose median sits far below the others - it would fail
nearly every row and vote "Hard" throughout, contributing no signal.

Nothing here re-runs a model, so you can change `THRESHOLD_MODE` in Cell 3 and
re-run just this cell and the next.

In [9]:
pooled = [preds[s["name"]][r["id"]]["chrf"] for r in sample for s in MODELS]
pooled_sorted = sorted(pooled)

def pct(p):
    return pooled_sorted[min(len(pooled_sorted) - 1, int(p * len(pooled_sorted)))]

print("chrF++ distribution per model:")
print("  {:<10} {:>7} {:>7} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for s in MODELS:
    v = sorted(x["chrf"] for x in preds[s["name"]].values())
    print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
        s["name"], v[len(v) // 4], v[len(v) // 2], v[3 * len(v) // 4],
        sum(v) / len(v)))
print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
    "POOLED", pct(.25), pct(.50), pct(.75), sum(pooled) / len(pooled)))

print("\nmodel-free floors (from Cell 4):")
print("  copy the Hindi source       : {:.1%}".format(COPY_FLOOR))
print("  unrelated {:<9} text     : {:.1%}".format(TARGET_LANG, RAND_FLOOR))
print("  binding floor (the higher)  : {:.1%}  <- what the threshold must clear".format(
    BINDING_FLOOR))


def difficulty_at(th):
    out = Counter()
    for r in sample:
        votes = sum(preds[s["name"]][r["id"]]["chrf"] >= th for s in MODELS)
        out["Easy" if votes == 3 else ("Medium" if votes == 2 else "Hard")] += 1
    return out


# ---- pick the threshold ----
if THRESHOLD_MODE == "auto_median":
    THRESHOLD = pct(.50)
    why = "median of all pooled model scores"
elif THRESHOLD_MODE == "floor_margin":
    THRESHOLD = BINDING_FLOOR + FLOOR_MARGIN
    why = "binding floor + {:.2f}".format(FLOOR_MARGIN)
elif THRESHOLD_MODE == "fixed":
    THRESHOLD = FIXED_THRESHOLD
    why = "FIXED_THRESHOLD from Cell 3"
else:
    raise ValueError("unknown THRESHOLD_MODE: " + str(THRESHOLD_MODE))

print("\nsensitivity - what each threshold would produce:")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
cands = sorted(set([round(x, 3) for x in
                    [.30, .35, .40, .45, .50, .55, .60, .65, .70,
                     round(COPY_FLOOR, 3), round(THRESHOLD, 3)]]))
for th in cands:
    d = difficulty_at(th)
    tag = ""
    if abs(th - round(THRESHOLD, 3)) < 1e-9:
        tag += "  <- CHOSEN"
    if abs(th - round(COPY_FLOOR, 3)) < 1e-9:
        tag += "  (copy-input floor)"
    print("  {:>9.3f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), tag))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))

if THRESHOLD <= BINDING_FLOOR:
    print("  WARNING: at or below the binding floor - a model could pass without")
    print("  really translating. Use THRESHOLD_MODE='floor_margin' or 'fixed'.")
else:
    print("  sits {:.1f} points above the binding floor - OK".format(
        100 * (THRESHOLD - BINDING_FLOOR)))

if THRESHOLD >= 0.95:
    print("  WARNING: the threshold sits at the very top of the range. More")
    print("  than half of all scores are near-perfect, so this demands an")
    print("  almost exact match and Easy becomes unreachable. Set")
    print("  THRESHOLD_MODE='fixed' with a value from the table above.")

_bands = difficulty_at(THRESHOLD)
if min(_bands.get(k, 0) for k in ("Easy", "Medium", "Hard")) == 0:
    print("  WARNING: one difficulty band is empty at this threshold - the")
    print("  split carries little information. Pick another value.")

chrF++ distribution per model:
  model          p25  median     p75    mean
  mistral     26.0%   35.4%   44.9%   36.6%
  llama       31.3%   42.5%   54.4%   43.2%
  gemma       30.5%   41.4%   53.1%   42.1%
  POOLED      28.6%   39.2%   52.3%   40.7%

model-free floors (from Cell 4):
  copy the Hindi source       : 36.4%
  unrelated Sanskrit  text     : 9.6%
  binding floor (the higher)  : 36.4%  <- what the threshold must clear

sensitivity - what each threshold would produce:
  threshold    Easy  Medium   Hard
      0.300     161      63     76
      0.350     124      68    108
      0.364     112      69    119  (copy-input floor)
      0.392      87      64    149  <- CHOSEN
      0.400      85      61    154
      0.450      56      51    193
      0.500      39      43    218
      0.550      22      31    247
      0.600      16      16    268
      0.650       9      11    280
      0.700       6       6    288

THRESHOLD = 0.392  (median of all pooled model scores)
  sits 2.

### Cell 10 - Apply the threshold and write the schema

Each model votes 1 where its chrF++ clears `THRESHOLD`; the votes sum into
Easy / Medium / Hard exactly as in every other split.

Output rows are rebuilt key-by-key from `SCHEMA_KEYS`, so the file carries
exactly the 14 IndicSample fields in schema order. `difficulty` is the only
value that changes unless you set `SET_EVAL_METRIC` in Cell 3. Translations and
raw scores go to the audit file.

In [10]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results = []
audit = []

for row in sample:
    scores = [preds[s["name"]][row["id"]]["chrf"] for s in MODELS]
    votes  = [int(x >= THRESHOLD) for x in scores]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":           row["id"],
        "difficulty":   difficulty,
        "votes":        votes,
        "chrf":         [round(x, 4) for x in scores],
        "threshold":    round(THRESHOLD, 4),
        "dataset":      DATASET,
        "subcategory":  row["subcategory"],
        "hindi":        " ".join(row["question"].split()),
        "target_lang":  TARGET_LANG,
        "reference":    " ".join(str(row["answer"]).split()),
        "translations": {s["name"]: preds[s["name"]][row["id"]]["output"]
                         for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))
print("threshold used: {:.3f}".format(THRESHOLD))

Saved -> vedic_hi_sa_difficulty.jsonl (300 rows)
Audit -> vedic_hi_sa_audit.jsonl
threshold used: 0.392


### Cell 11 - Verify and report

Checks before trusting the file:

1. **Schema** - all 14 keys in order, no nulls in `difficulty`.
2. **Difficulty distribution.**
3. **Per-model mean chrF++ against the unrelated-translation floor.** A model
   at or below that floor is emitting Tamil that has nothing to do with the
   source - check the audit file before concluding anything about difficulty.
4. **Wrong-script rate** (`hi_kn` only). A model answering in Devanagari has
   not translated at all. For `hi_sa` this check cannot apply - both sides use
   Devanagari - which is exactly why the copy-the-input floor matters more
   there.
5. **Empty-output rate**, same reasoning.

**Expect a Hard-heavy result, and read it carefully.** None of these three
models is strong at Sanskrit, and Tamil is only moderately resourced for them.
A Hard skew here may say more about the models than about the rows - which is a
legitimate finding, but only if the Devanagari and empty rates are low. If they
are high, fix the prompt before trusting the labels.

The sample rows at the end print the source, the reference, and all three
translations, which is the fastest way to sanity-check that Hard rows are
genuinely hard.

In [11]:
# 1. schema integrity
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))

# 2. difficulty distribution
dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

# 3. per-model score vs the do-nothing floor
print("\nMean chrF++:")
for s in MODELS:
    v = [x["chrf"] for x in preds[s["name"]].values()]
    m = sum(v) / len(v)
    flag = "  <- at/below the binding floor" if m <= BINDING_FLOOR else ""
    print("  {:<10} {:.1%}{}".format(s["name"], m, flag))
print("  {:<10} {:.1%}  <- unrelated {} floor".format("unrelated", RAND_FLOOR, TARGET_LANG))
print("  {:<10} {:.1%}  <- copy the Hindi input".format("copy", COPY_FLOOR))
print("  {:<10} {:.1%}  <- binding floor".format("binding", BINDING_FLOOR))

# 4. empty or truncated outputs
print("\nOutput health:")
for s in MODELS:
    v = list(preds[s["name"]].values())
    empty = sum(1 for x in v if x["n_words"] == 0)
    deva  = sum(x.get("wrong_script", 0) for x in v)
    flags = []
    if deva / total > 0.10:
        flags.append("wrong script")
    if empty / total > 0.05:
        flags.append("empty outputs")
    print("  {:<10} empty {:>3}/{} | wrong-script {:>3}/{} | median words {}{}".format(
        s["name"], empty, total, deva, total,
        sorted(x["n_words"] for x in v)[len(v) // 2],
        "  <- " + ", ".join(flags) if flags else ""))

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] chrf={}".format(a["id"], a["difficulty"], a["chrf"]))
    print("    hindi    :", a["hindi"][:88])
    print("    reference:", a["reference"][:88])
    for k, v in a["translations"].items():
        print("    {:<8} :".format(k), v[:88])

Schema check : 300 rows | wrong keys: 0 | null difficulty: 0

Difficulty distribution (threshold 0.392):
  Easy   :   86  (28.7%)
  Medium :   65  (21.7%)
  Hard   :  149  (49.7%)

Mean chrF++:
  mistral    36.6%
  llama      43.2%
  gemma      42.1%
  unrelated  9.6%  <- unrelated Sanskrit floor
  copy       36.4%  <- copy the Hindi input
  binding    36.4%  <- binding floor

Output health:
  mistral    empty   0/300 | wrong-script   0/300 | median words 7
  llama      empty   0/300 | wrong-script   0/300 | median words 7
  gemma      empty   0/300 | wrong-script   0/300 | median words 7

--- 2 sample rows ---

  vedic_002727 [Easy] chrf=[0.6635, 0.6606, 0.6741]
    hindi    : संहितायां स्वरिताद्‌ अनुदात्तानाम्‌ एकश्रुतिः स्याद ये सूत्र में आये पदों का अन्वय है।
    reference: संहितायां स्वरिताद्‌ अनुदात्तानाम्‌ एकश्रुतिः स्यादिति सूत्रगतपदानाम्‌ अन्वयः।
    mistral  : स्वरिताद्‌ अनुदात्तानाम्‌ एकश्रुतिः स्यात् ये सूत्राणि पदानाम्‌ अन्वयः।
    llama    : संहितायां स्वरिताद्‌ अनुदात्ता